### Random forest MTL na reprezentacji fingerprint - zestaw 3 - Kardiotoksyczność**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. hERG (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. VDss (Lombardo)
5. AMES Mutagenicity

Wyniki dla STL:

In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_folder = "/content/drive/MyDrive/data_splits"

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [ ]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer):
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                fingerprints.append(np.zeros(self.length))

            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [6]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: 
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
def train_hybrid_mtl_rf(X_train, y_train_dict, X_test, y_test_dict, reg_tasks, class_tasks):
    all_tasks = reg_tasks + class_tasks


    Y_train_raw = np.hstack([y_train_dict[task] for task in all_tasks])
    Y_test_raw = np.hstack([y_test_dict[task] for task in all_tasks])

    Y_train_imputed = Y_train_raw.copy()
    Y_test_imputed = Y_test_raw.copy()

    print(">> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...")
    for i, task in enumerate(all_tasks):
        known_train_idx = ~np.isnan(Y_train_raw[:, i])
        missing_train_idx = np.isnan(Y_train_raw[:, i])
        missing_test_idx = np.isnan(Y_test_raw[:, i])

        if task in reg_tasks:
            model_single = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i])
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict(X_train[missing_train_idx])
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict(X_test[missing_test_idx])

        elif task in class_tasks:
            model_single = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i].astype(int))
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict_proba(X_train[missing_train_idx])[:, 1]
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict_proba(X_test[missing_test_idx])[:, 1]

    print(">> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...")
    mtl_model = RandomForestRegressor(n_estimators=1000, max_depth=25, max_features="sqrt", random_state=42, n_jobs=-1)
    mtl_model.fit(X_train, Y_train_imputed)

    Y_pred = mtl_model.predict(X_test)

    metrics_summary = {}
    for i, task in enumerate(all_tasks):
        known_test_idx = ~np.isnan(Y_test_raw[:, i])
        true_y = Y_test_raw[known_test_idx, i]
        pred_y = Y_pred[known_test_idx, i]

        if task in reg_tasks:
            metrics_summary[task] = {
                "task_type": "regression",  
                "test_metrics": {
                    "rmse": np.sqrt(mean_squared_error(true_y, pred_y)),
                    "mae": mean_absolute_error(true_y, pred_y),
                    "r2": r2_score(true_y, pred_y)
                }
            }
        elif task in class_tasks:
            pred_classes = (pred_y >= 0.5).astype(int)
            metrics_summary[task] = {
                "task_type": "classification",  
                "test_metrics": {
                    "accuracy": accuracy_score(true_y, pred_classes),
                    "f1": f1_score(true_y, pred_classes, zero_division=0),
                    "auroc": roc_auc_score(true_y, pred_y)
                }
            }

    return mtl_model, metrics_summary

# Test 1: hERG (Wang) + Lipophilicity (AstraZeneca)

In [ ]:

import os

reg_tasks = ['Lipophilicity_AZ']
class_tasks = ['hERG'] 

all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")

train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_herg, test_herg = load_split_pickle('hERG')

df_train_list = [train_lipo, train_herg]
df_test_list = [test_lipo, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="herg + Lipophilicity"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[10:01:46] WARNING: not removing hydrogen atom without neighbors
[10:01:48] WARNING: not removing hydrogen atom without neighbors
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use MorganGenerator
[10:01:48] DEPRECATION WARNING: please use M

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AZ (REGRESSION)

  RMSE     : 0.8981
  MAE      : 0.6891
  R²       : 0.4541

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8397
  F1       : 0.9014
  AUROC    : 0.7979

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test2: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB)

In [ ]:


reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['hERG']  
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_herg, test_herg = load_split_pickle('hERG')

df_train_list = [train_lipo, train_sol, train_herg]
df_test_list = [test_lipo, test_sol, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

   
    print_metrics(task_data, task=task_type)


    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:03:21] DEPRECATION WARNING: please use MorganGenerator
[10:0

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AZ (REGRESSION)

  RMSE     : 0.9382
  MAE      : 0.7290
  R²       : 0.4042

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4883
  MAE      : 1.1405
  R²       : 0.5918

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8473
  F1       : 0.9029
  AUROC    : 0.8112

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test3: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo)

In [ ]:


reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_herg, test_herg = load_split_pickle('hERG')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

df_train_list = [train_lipo, train_sol, train_vdss, train_herg]
df_test_list = [test_lipo, test_sol, test_vdss, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + VDss_Lombardo"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:07:46] DEPRECATION WARNING: please use MorganGenerator
[10:0

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AZ (REGRESSION)

  RMSE     : 1.0126
  MAE      : 0.7995
  R²       : 0.3060

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.6171
  MAE      : 1.2666
  R²       : 0.5181

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 9.8217
  MAE      : 4.2188
  R²       : -1.0593

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8397
  F1       : 0.8976
  AUROC    : 0.8319

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test4: hERG (Wang) + VDss (Lombardo)

In [ ]:


reg_tasks = ['VDss_Lombardo']
class_tasks = ['hERG'] 
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

df_train_list = [train_vdss, train_herg]
df_test_list = [test_vdss, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + VDss_Lombardo"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


[10:12:15] WARNING: not removing hydrogen atom without neighbors
[10:12:15] WARNING: not removing hydrogen atom without neighbors
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use MorganGenerator
[10:12:15] DEPRECATION WARNING: please use M

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 5.8833
  MAE      : 3.3689
  R²       : 0.2611

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8397
  F1       : 0.9005
  AUROC    : 0.8211

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test5: hERG (Wang) + Solubility (AqSolDB)

In [ ]:


reg_tasks = ['Solubility_AqSolDB']
class_tasks = ['hERG'] 
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

df_train_list = [train_sol, train_herg]
df_test_list = [test_sol, test_herg]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + Solubility (AqSolDB)"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:12:50] DEPRECATION WARNING: please use MorganGenerator
[10:1

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4718
  MAE      : 1.1261
  R²       : 0.6008

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.7710
  F1       : 0.8469
  AUROC    : 0.7948

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test6: Solubility (AqSolDB) + VDss (Lombardo)

In [ ]:


reg_tasks = ['Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = [] 
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

df_train_list = [train_sol, train_vdss]
df_test_list = [test_sol, test_vdss]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Solubility (AqSolDB) + VDss (Lombardo)"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:15:47] DEPRECATION WARNING: please use MorganGenerator
[10:1

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.5856
  MAE      : 1.2337
  R²       : 0.5367

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 9.6091
  MAE      : 4.1636
  R²       : -0.9711

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test7: hERG (Wang) + AMES Mutagenicity

In [ ]:


reg_tasks = []
class_tasks = ['hERG', 'AMES'] 
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')

df_train_list = [train_herg, train_ames]
df_test_list = [test_herg, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")


    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG + AMES Mutagenicity"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:18:56] DEPRECATION WARNING: please use MorganGenerator
[10:1

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8168
  F1       : 0.8812
  AUROC    : 0.8088

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.8281
  F1       : 0.8399
  AUROC    : 0.9034

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt


# Test8: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo) + AMES Mutagenicity

In [ ]:


reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG', 'AMES'] 
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_kardiotoksycznosc_fingerprints_rf.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

print(">> Ładowanie danych z plików pickle...")
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')

df_train_list = [train_lipo, train_sol, train_vdss, train_herg, train_ames]
df_test_list = [test_lipo, test_sol, test_vdss, test_herg, test_ames]

print(">> Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

dirname = os.path.dirname(filepath)
if dirname and not os.path.exists(dirname):
    os.makedirs(dirname)

print("\n>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")

    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo) + AMES Mutagenicity"
    )

print(f"[SUCCESS] Proces zakończony. Wyniki dopisano do: {filepath}")

>> Ładowanie danych z plików pickle...
>> Przygotowywanie zintegrowanych zbiorów danych MTL...


Streaming output truncated to the last 5000 lines.
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:19:54] DEPRECATION WARNING: please use MorganGenerator
[10:1

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> GENEROWANIE RAPORTÓW KOŃCOWYCH <<<
Endpoint: Lipophilicity_AZ (REGRESSION)

  RMSE     : 1.0213
  MAE      : 0.8092
  R²       : 0.2940

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.6299
  MAE      : 1.2768
  R²       : 0.5104

Endpoint: VDss_Lombardo (REGRESSION)

  RMSE     : 12.0398
  MAE      : 4.5763
  R²       : -2.0944

Endpoint: hERG (CLASSIFICATION)

  Accuracy : 0.8321
  F1       : 0.8922
  AUROC    : 0.8323

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7256
  F1       : 0.7011
  AUROC    : 0.8452

[SUCCESS] Proces zakończony. Wyniki dopisano do: /content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints_rf.txt
